In [1]:
import os
from sodapy import Socrata
import requests
from bs4 import BeautifulSoup
import time

from socrata_interface.domain import Domain

In [ ]:
SCROLL_FILE = "last_scroll_id.txt"
DOMAIN_FILE = "socrata_domains.txt"

def load_state():
    """Load scroll ID + seen domains if resuming."""
    # Load scroll ID
    if os.path.exists(SCROLL_FILE):
        with open(SCROLL_FILE, "r") as f:
            scroll_id = f.read().strip()
            if scroll_id == "":
                scroll_id = "*"   # fallback
    else:
        scroll_id = "*"

    # Load seen domains set
    seen = set()
    if os.path.exists(DOMAIN_FILE):
        with open(DOMAIN_FILE, "r") as f:
            for line in f:
                seen.add(line.strip())

    return scroll_id, seen


def save_scroll_id(scroll_id):
    """Write latest scroll ID to disk so we can resume."""
    with open(SCROLL_FILE, "w") as f:
        f.write(scroll_id)


def get_all_domains_resume():
    url = "https://api.us.socrata.com/api/catalog" # Seems to be a catalog of all things accessible by the api
    limit = 1000 # Limit 1000 because it is small enough to avoid timeouts. 10000 gets timed out. Optimal would probably be between

    # Load previous state
    scroll_id, seen = load_state()

    print(f"Starting with scroll_id={scroll_id}, {len(seen)} domains already saved.")

    # Open output file in append mode
    with open(DOMAIN_FILE, "a") as f_out:

        while True:
            print(f"Fetching scroll_id={scroll_id} ...")

            params = {"scroll_id": scroll_id, "limit": limit}

            try:
                resp = requests.get(url, params=params, timeout=10)
                resp.raise_for_status()
            except Exception as e:
                print(f"Error: {e}, retrying in 5 seconds...")
                time.sleep(5)
                continue

            data = resp.json()
            results = data.get("results", [])

            if not results:
                print("Deep scroll completed or no more results.")
                break

            # Process results
            for item in results:
                metadata = item.get("metadata", {})
                domain = metadata.get("domain")

                if domain and domain not in seen:
                    f_out.write(domain + "\n")
                    f_out.flush()  # ensure immediate write
                    seen.add(domain)

            # Update scroll ID for next request
            next_scroll = results[-1].get("resource").get("id") # id of previous resource can be used to get next scroll
            if not next_scroll:
                print("Finished scrolling dataset.")
                break

            scroll_id = next_scroll
            save_scroll_id(scroll_id)  # persist checkpoint

            time.sleep(0.2) # Only to avoid timeouts, may not be necessary

    print(f"\nCompleted with {len(seen)} total domains.")
    return seen

# Getting all domains to find cities, as the original study was about the state of urban data across US cities, not just NY
domains = get_all_domains_resume() # 555 domains discovered amongst ~220k things in the catalog, but many are for same entity

Starting with scroll_id=*, 118 domains already saved.
Fetching scroll_id=* ...
Fetching scroll_id=26is-s4fm ...
Fetching scroll_id=2auq-ndkr ...
Fetching scroll_id=2fh6-vrts ...
Fetching scroll_id=2k8a-dz2p ...
Fetching scroll_id=2qsi-qheg ...
Fetching scroll_id=2vhj-s442 ...
Fetching scroll_id=322a-riji ...
Fetching scroll_id=36ib-rtmu ...
Fetching scroll_id=3b78-mfyi ...
Fetching scroll_id=3fxc-nque ...
Fetching scroll_id=3kbj-ypat ...
Fetching scroll_id=3qys-jk4f ...
Fetching scroll_id=3vft-99rh ...
Fetching scroll_id=4292-pktu ...
Fetching scroll_id=46zs-4ngp ...
Fetching scroll_id=4bn5-jdm8 ...
Fetching scroll_id=4g6s-ak9g ...
Fetching scroll_id=4m2v-hzec ...
Fetching scroll_id=4rka-uupg ...
Fetching scroll_id=4w6e-7nqk ...
Fetching scroll_id=52ny-36z2 ...
Fetching scroll_id=576d-v5m3 ...
Fetching scroll_id=5bn2-vnxz ...
Fetching scroll_id=5g3x-yfbg ...
Fetching scroll_id=5kp7-t9c8 ...
Fetching scroll_id=5rei-mff9 ...
Fetching scroll_id=5w2u-reag ...
Fetching scroll_id=62vi-89fw .

In [7]:
def filter_lines(input_path: str, output_path: str, include: str | None = None, exclude: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # If include phrase is given, skip lines that don't contain it
            if include is not None and include not in line:
                continue
            
            # If exclude phrase is given, skip lines that DO contain it
            if exclude is not None and exclude in line:
                continue
            
            outfile.write(line)


In [11]:
def filter_lines_start(input_path: str, output_path: str, include_start: str | None = None, exclude_start: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # Check include-start condition
            if include_start is not None and not line.startswith(include_start):
                continue

            # Check exclude-start condition
            if exclude_start is not None and line.startswith(exclude_start):
                continue

            outfile.write(line)

In [16]:
filter_lines_start("socrata_domains.txt", "socrata_domains_data.txt", include_start="data.")

In [17]:
filter_lines("socrata_domains_data.txt", "socrata_domains_countyless.txt", exclude="county")

In [ ]:
def normalize_url(url: str) -> str:
    if not url.startswith(("http://", "https://")):
        return "https://" + url
    return url

def find_city(input_path: str, output_path: str, phrase: str): # Didn't even get all the cities
    print("Not city:")
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            text = ""
            response = requests.get(normalize_url(line.strip()))
            
            if response.ok:

                soup = BeautifulSoup(response.text, "html.parser")

                # Try meta name="title"
                meta_title = soup.find("meta", attrs={"name": "title"}) # Probably need more checks to properly find cities
                if meta_title is not None and "content" in meta_title.attrs:
                    text = meta_title["content"]

                # Fallback to the <title> tag
                elif soup.title:
                    text = soup.title.text.strip()

            if phrase in text.lower():
                outfile.write(line)
            else:
                print(line)

In [32]:
find_city("socrata_domains_countyless.txt", "socrata_domains_cities_only.txt", phrase="city")

Not city:
data.cityofnewyork.us

data.auburnwa.gov

data.cambridgema.gov

data.novascotia.ca

data.honolulu.gov

data.bayareametro.gov

data.cdc.gov

data.calgary.ca

data.bts.gov

data.pa.gov

data.edmonton.ca

data.dumfriesva.gov

data.cityofgainesville.org

data.delaware.gov

data.oce.pr.gov

data.kcmo.org

data.cincinnati-oh.gov

data.buffalony.gov

data.mmcp.ms.gov

data.wcad.org

data.memphistn.gov

data.readingpa.gov

data.framinghamma.gov

data.smcgov.org

data.vermont.gov

data.datacenterresearch.org

data.tompsc.com

data.orcities.org

data.stocktonca.gov

data.cstx.gov

data.coloradosprings.gov

data.sfgov.org

data.nhitc.org

data.fortworthtexas.gov

data.qac.org

data.oxnard.org



# Metadata needed

- Schema: "columns" | list of columns 

- Column Types: "dataTypeName" | in the list of columns

- Column Names: "name" or "fieldName" | in the list of columns

- Zipcode: "the_geom"

- Nulls: "non_null" and "null" | https://<domain>/resource/<dataset_id>.json?$select=count(*)&$where=<column_name>%20IS%20NULL

- Category: "category" | for top categories of each city

- Format: "displayType" or "viewType"

- Number of Rows: https://<domain>/resource/<dataset_id>.json?$select=count(*)

- Tags: "tags" | list of tags

- Number of Downloads: "downloadCount"

- Number of Views: "viewCount"

- Age of Dataset: "createdAt"

- Update Frequency: "indexUpdatedAt" or "rowsUpdatedAt"

In [2]:
nola = Socrata('data.nola.gov', None)

In [ ]:
metadata = nola.get_metadata("2jgv-pqrq")

cols = metadata.get("columns") or []
relevant_columns = [
    {
        "name": c.get("fieldName"), 
        "type": c.get("dataTypeName"),
    }
    for c in cols if c.get("fieldName")
]

def _quote_field_name(field):
        """
        Quote field names that need it (contain special characters).
        """
        special_chars = {':', '@', '-', ' ', '.', '/', '\\', '(', ')'}
        if any(c in field for c in special_chars):
            escaped = field.replace('`', '``')
            return f"`{escaped}`"
        return field

select_parts = []
for col in cols:
    field = col["fieldName"]
    dtype = col["dataTypeName"]
    
    quoted_field = _quote_field_name(field)
    safe_alias = field.replace(":", "_").replace("@", "_").replace("-", "_")
    
    # null count
    select_parts.append(
        f"(count(*) - count({quoted_field})) AS {safe_alias}_nulls"
    )
    
    # Simplified semantic nulls for text fields only
    TEXT_LIKE_TYPES = {"text", "url", "email", "phone", "html"}
    if dtype in TEXT_LIKE_TYPES:
        # Simpler check - just trim and empty string
        semantic = (
            f"sum(CASE WHEN "
            f"{quoted_field} IS NULL OR "
            f"trim({quoted_field}) = '' "
            f"THEN 1 ELSE 0 END) AS {safe_alias}_semantic_nulls"
        )
    else:
        semantic = f"0 AS {safe_alias}_semantic_nulls"
    select_parts.append(semantic)

In [10]:
metadata['columns']

[{'id': 541844684,
  'name': 'Service Request #',
  'dataTypeName': 'text',
  'description': '',
  'fieldName': 'service_request',
  'position': 1,
  'renderTypeName': 'text',
  'tableColumnId': 75233504,
  'width': 100,
  'cachedContents': {'non_null': '967468',
   'largest': '2026-1257280',
   'null': '0',
   'top': [{'item': '101000002301', 'count': '1'},
    {'item': '101000002302', 'count': '1'},
    {'item': '101000002304', 'count': '1'},
    {'item': '101000002313', 'count': '1'},
    {'item': '101000002314', 'count': '1'},
    {'item': '101000002316', 'count': '1'},
    {'item': '101000002320', 'count': '1'},
    {'item': '101000002322', 'count': '1'},
    {'item': '101000002323', 'count': '1'},
    {'item': '101000002324', 'count': '1'},
    {'item': '101000002329', 'count': '1'},
    {'item': '101000002332', 'count': '1'},
    {'item': '101000002341', 'count': '1'},
    {'item': '101000002343', 'count': '1'},
    {'item': '101000002344', 'count': '1'},
    {'item': '101000002

In [4]:
select_parts

['(count(*) - count(service_request)) AS service_request_nulls',
 "sum(CASE WHEN service_request IS NULL OR trim(service_request) = '' THEN 1 ELSE 0 END) AS service_request_semantic_nulls",
 '(count(*) - count(request_type)) AS request_type_nulls',
 "sum(CASE WHEN request_type IS NULL OR trim(request_type) = '' THEN 1 ELSE 0 END) AS request_type_semantic_nulls",
 '(count(*) - count(request_reason)) AS request_reason_nulls',
 "sum(CASE WHEN request_reason IS NULL OR trim(request_reason) = '' THEN 1 ELSE 0 END) AS request_reason_semantic_nulls",
 '(count(*) - count(date_created)) AS date_created_nulls',
 '0 AS date_created_semantic_nulls',
 '(count(*) - count(date_modified)) AS date_modified_nulls',
 '0 AS date_modified_semantic_nulls',
 '(count(*) - count(case_close_date)) AS case_close_date_nulls',
 '0 AS case_close_date_semantic_nulls',
 '(count(*) - count(request_status)) AS request_status_nulls',
 "sum(CASE WHEN request_status IS NULL OR trim(request_status) = '' THEN 1 ELSE 0 END) 

In [54]:
', '.join(select_parts[-16:])

'(count(*) - count(`:@computed_region_ewbu_t8bu`)) AS :@computed_region_ewbu_t8bu_nulls, 0 AS :@computed_region_ewbu_t8bu_semantic_nulls, (count(*) - count(`:@computed_region_k37d_then`)) AS :@computed_region_k37d_then_nulls, 0 AS :@computed_region_k37d_then_semantic_nulls, (count(*) - count(`:@computed_region_m56f_hbma`)) AS :@computed_region_m56f_hbma_nulls, 0 AS :@computed_region_m56f_hbma_semantic_nulls, (count(*) - count(`:@computed_region_7fw3_kdpf`)) AS :@computed_region_7fw3_kdpf_nulls, 0 AS :@computed_region_7fw3_kdpf_semantic_nulls, (count(*) - count(`:@computed_region_spev_d8jm`)) AS :@computed_region_spev_d8jm_nulls, 0 AS :@computed_region_spev_d8jm_semantic_nulls, (count(*) - count(`:@computed_region_sikx_bdeb`)) AS :@computed_region_sikx_bdeb_nulls, 0 AS :@computed_region_sikx_bdeb_semantic_nulls, (count(*) - count(`:@computed_region_evki_aju8`)) AS :@computed_region_evki_aju8_nulls, 0 AS :@computed_region_evki_aju8_semantic_nulls, (count(*) - count(`:@computed_region_u4y

In [ ]:
'(count(*) - count(Location)) AS location_nulls'

In [45]:
nola.get("2jgv-pqrq", select = "geocoded_column")

[{'geocoded_column': {'latitude': '0.0', 'longitude': '0.0'}},
 {'geocoded_column': {'latitude': '0.0', 'longitude': '0.0'}},
 {'geocoded_column': {'latitude': '29.978768001649772',
   'longitude': '-90.12458247242685'}},
 {'geocoded_column': {'latitude': '29.940758205316882',
   'longitude': '-90.091290391251'}},
 {'geocoded_column': {'latitude': '29.988242136809447',
   'longitude': '-90.10896265604138'}},
 {'geocoded_column': {'latitude': '29.97462748129893',
   'longitude': '-90.05021192917715'}},
 {'geocoded_column': {'latitude': '29.963911760299002',
   'longitude': '-90.02904855115517'}},
 {'geocoded_column': {'latitude': '29.96436285930971',
   'longitude': '-90.05677324981151'}},
 {'geocoded_column': {'latitude': '0.0', 'longitude': '0.0'}},
 {'geocoded_column': {'latitude': '29.950571578449225',
   'longitude': '-90.10464085367344'}},
 {'geocoded_column': {'latitude': '29.940758205316882',
   'longitude': '-90.091290391251'}},
 {'geocoded_column': {'latitude': '29.93253310222

In [55]:
nola.get("2jgv-pqrq", select = ', '.join(select_parts[-16:]))

HTTPError: 400 Client Error: Bad Request.
	Could not parse SoQL query "select (count(*) - count(`:@computed_region_ewbu_t8bu`)) AS :@computed_region_ewbu_t8bu_nulls, 0 AS :@computed_region_ewbu_t8bu_semantic_nulls, (count(*) - count(`:@computed_region_k37d_then`)) AS :@computed_region_k37d_then_nulls, 0 AS :@computed_region_k37d_then_semantic_nulls, (count(*) - count(`:@computed_region_m56f_hbma`)) AS :@computed_region_m56f_hbma_nulls, 0 AS :@computed_region_m56f_hbma_semantic_nulls, (count(*) - count(`:@computed_region_7fw3_kdpf`)) AS :@computed_region_7fw3_kdpf_nulls, 0 AS :@computed_region_7fw3_kdpf_semantic_nulls, (count(*) - count(`:@computed_region_spev_d8jm`)) AS :@computed_region_spev_d8jm_nulls, 0 AS :@computed_region_spev_d8jm_semantic_nulls, (count(*) - count(`:@computed_region_sikx_bdeb`)) AS :@computed_region_sikx_bdeb_nulls, 0 AS :@computed_region_sikx_bdeb_semantic_nulls, (count(*) - count(`:@computed_region_evki_aju8`)) AS :@computed_region_evki_aju8_nulls, 0 AS :@compu

In [43]:
nola.get("2jgv-pqrq", select = 
        """
        (count(*) - count(service_request)) AS service_request_nulls,
        sum(CASE WHEN service_request IS NULL OR trim(service_request) = '' THEN 1 ELSE 0 END) AS service_request_semantic_nulls,
        (count(*) - count(request_type)) AS request_type_nulls,
        sum(CASE WHEN request_type IS NULL OR trim(request_type) = '' THEN 1 ELSE 0 END) AS request_type_semantic_nulls,
        (count(*) - count(request_reason)) AS request_reason_nulls,
        sum(CASE WHEN request_reason IS NULL OR trim(request_reason) = '' THEN 1 ELSE 0 END) AS request_reason_semantic_nulls,
        (count(*) - count(date_created)) AS date_created_nulls,
        0 AS date_created_semantic_nulls,
        (count(*) - count(date_modified)) AS date_modified_nulls,
        0 AS date_modified_semantic_nulls,
        (count(*) - count(case_close_date)) AS case_close_date_nulls,
        0 AS case_close_date_semantic_nulls,
        (count(*) - count(request_status)) AS request_status_nulls,
        sum(CASE WHEN request_status IS NULL OR trim(request_status) = '' THEN 1 ELSE 0 END) AS request_status_semantic_nulls,
        (count(*) - count(responsible_agency)) AS responsible_agency_nulls,
        sum(CASE WHEN responsible_agency IS NULL OR trim(responsible_agency) = '' THEN 1 ELSE 0 END) AS responsible_agency_semantic_nulls,
        (count(*) - count(final_address)) AS final_address_nulls,
        sum(CASE WHEN final_address IS NULL OR trim(final_address) = '' THEN 1 ELSE 0 END) AS final_address_semantic_nulls,
        (count(*) - count(address_councildis)) AS address_councildis_nulls,
        sum(CASE WHEN address_councildis IS NULL OR trim(address_councildis) = '' THEN 1 ELSE 0 END) AS address_councildis_semantic_nulls,
        (count(*) - count(status)) AS status_nulls,
        sum(CASE WHEN status IS NULL OR trim(status) = '' THEN 1 ELSE 0 END) AS status_semantic_nulls,
        (count(*) - count(contractor)) AS contractor_nulls,
        sum(CASE WHEN contractor IS NULL OR trim(contractor) = '' THEN 1 ELSE 0 END) AS contractor_semantic_nulls,
        (count(*) - count(contractor_action)) AS contractor_action_nulls,
        sum(CASE WHEN contractor_action IS NULL OR trim(contractor_action) = '' THEN 1 ELSE 0 END) AS contractor_action_semantic_nulls,
        (count(*) - count(rowid)) AS rowid_nulls,
        0 AS rowid_semantic_nulls,
        (count(*) - count(final_x)) AS final_x_nulls,
        0 AS final_x_semantic_nulls,
        (count(*) - count(final_y)) AS final_y_nulls,
        0 AS final_y_semantic_nulls,
        (count(*) - count(longitude)) AS longitude_nulls,
        0 AS longitude_semantic_nulls,
        (count(*) - count(latitude)) AS latitude_nulls,
        0 AS latitude_semantic_nulls
        """)

[{'service_request_nulls': '0',
  'service_request_semantic_nulls': '0',
  'request_type_nulls': '1757',
  'request_type_semantic_nulls': '1757',
  'request_reason_nulls': '1778',
  'request_reason_semantic_nulls': '1778',
  'date_created_nulls': '0',
  'date_created_semantic_nulls': '0',
  'date_modified_nulls': '0',
  'date_modified_semantic_nulls': '0',
  'case_close_date_nulls': '228903',
  'case_close_date_semantic_nulls': '0',
  'request_status_nulls': '0',
  'request_status_semantic_nulls': '0',
  'responsible_agency_nulls': '6588',
  'responsible_agency_semantic_nulls': '6588',
  'final_address_nulls': '13205',
  'final_address_semantic_nulls': '13205',
  'address_councildis_nulls': '328991',
  'address_councildis_semantic_nulls': '328991',
  'status_nulls': '463342',
  'status_semantic_nulls': '463342',
  'contractor_nulls': '722534',
  'contractor_semantic_nulls': '722534',
  'contractor_action_nulls': '850469',
  'contractor_action_semantic_nulls': '850469',
  'rowid_nulls':

In [2]:
domain = Domain("data.nola.gov")

In [4]:
crungle = domain.metadata("2jgv-pqrq")

Initializing client for data.nola.gov...


In [5]:
crungle['columns']

[{'id': 541844684,
  'name': 'Service Request #',
  'dataTypeName': 'text',
  'description': '',
  'fieldName': 'service_request',
  'position': 1,
  'renderTypeName': 'text',
  'tableColumnId': 75233504,
  'width': 100,
  'cachedContents': {'non_null': '967468',
   'largest': '2026-1257280',
   'null': '0',
   'top': [{'item': '101000002301', 'count': '1'},
    {'item': '101000002302', 'count': '1'},
    {'item': '101000002304', 'count': '1'},
    {'item': '101000002313', 'count': '1'},
    {'item': '101000002314', 'count': '1'},
    {'item': '101000002316', 'count': '1'},
    {'item': '101000002320', 'count': '1'},
    {'item': '101000002322', 'count': '1'},
    {'item': '101000002323', 'count': '1'},
    {'item': '101000002324', 'count': '1'},
    {'item': '101000002329', 'count': '1'},
    {'item': '101000002332', 'count': '1'},
    {'item': '101000002341', 'count': '1'},
    {'item': '101000002343', 'count': '1'},
    {'item': '101000002344', 'count': '1'},
    {'item': '101000002

In [6]:
domain.null_counts("2jgv-pqrq", crungle['columns'])

[{'service_request_nulls': '0',
  'service_request_semantic_nulls': '0',
  'request_type_nulls': '1757',
  'request_type_semantic_nulls': '1757',
  'request_reason_nulls': '1778',
  'request_reason_semantic_nulls': '1778',
  'date_created_nulls': '0',
  'date_modified_nulls': '0',
  'case_close_date_nulls': '229117',
  'request_status_nulls': '0',
  'request_status_semantic_nulls': '0',
  'responsible_agency_nulls': '6588',
  'responsible_agency_semantic_nulls': '6588',
  'final_address_nulls': '13205',
  'final_address_semantic_nulls': '13205',
  'address_councildis_nulls': '328991',
  'address_councildis_semantic_nulls': '328991',
  'status_nulls': '463342',
  'status_semantic_nulls': '463342',
  'contractor_nulls': '722665',
  'contractor_semantic_nulls': '722665',
  'contractor_action_nulls': '850733',
  'contractor_action_semantic_nulls': '850733',
  'rowid_nulls': '0',
  'final_x_nulls': '40876',
  'final_y_nulls': '40876',
  'longitude_nulls': '0',
  'latitude_nulls': '0',
  '__